In [1]:
import requests
import pandas as pd
import time
from datetime import datetime
from pathlib import Path

In [3]:
from pathlib import Path

BRONZE_PATH = Path("/home/jovyan/work/submission/output/bronze")
BRONZE_PATH.mkdir(parents=True, exist_ok=True)

print("Bronze folder ready:", BRONZE_PATH)

Bronze folder ready: /home/jovyan/work/submission/output/bronze


In [5]:
BRONZE_PATH = Path("/home/jovyan/work/submission/output/bronze")
BRONZE_PATH.mkdir(parents=True, exist_ok=True)

In [11]:
AUTH_URL = "http://mock-api:8000/api/v1/auth/token"

USERNAME = "candidate"
PASSWORD = "blue-owls-2026"

In [12]:
def get_token():
    response = requests.post(
        AUTH_URL,
        json={
            "username": USERNAME,
            "password": PASSWORD
        }
    )

    if response.status_code != 200:
        raise Exception(f"Failed to authenticate: {response.text}")

    token = response.json()["access_token"]
    return token

In [13]:
token = get_token()
print("Token acquired:", token[:30], "...")

Token acquired: eyJhbGciOiJIUzI1NiIsInR5cCI6Ik ...


In [14]:
def api_request(endpoint, params=None, max_retries=5):
    global token

    url = f"{BASE_URL}/data/{endpoint}"

    for attempt in range(max_retries):

        headers = {"Authorization": f"Bearer {token}"}

        response = requests.get(url, headers=headers, params=params)

        # SUCCESS
        if response.status_code == 200:
            return response.json()

        # TOKEN EXPIRED
        if response.status_code == 401:
            print("Token expired — refreshing...")
            token = get_token()
            continue

        # RETRYABLE ERRORS
        if response.status_code in [429, 500]:
            wait_time = 2 ** attempt
            print(f"Retrying in {wait_time}s (status {response.status_code})")
            time.sleep(wait_time)
            continue

        # OTHER ERRORS
        raise Exception(f"API error {response.status_code}: {response.text}")

    raise Exception("Max retries exceeded")

In [19]:
orders = api_request("orders", params={"page":1, "page_size":5})

records = orders["data"]

print(len(records))
print(records[0])

5
{'order_id': 'e481f51cbdc54678b7cc49136f2d6af7', 'customer_id': '9ef432eb6251297304e76186b10a928d', 'order_status': 'delivered', 'order_purchase_timestamp': '2017-10-02 10:56:33', 'order_approved_at': '2017-10-02 11:07:15', 'order_delivered_carrier_date': '2017-10-04 19:55:00', 'order_delivered_customer_date': '2017-10-10 21:25:13', 'order_estimated_delivery_date': '2017-10-18 00:00:00'}


In [16]:
BASE_URL = "http://mock-api:8000/api/v1"

In [18]:
orders = api_request("orders", params={"page":1, "page_size":5})

print(type(orders))
print(orders.keys())

<class 'dict'>
dict_keys(['data', 'pagination'])


In [20]:
ingest_endpoint("orders")

NameError: name 'ingest_endpoint' is not defined

In [24]:
def ingest_endpoint(endpoint, page_size=1000):

    all_records = []
    page = 1

    while True:

        try:
            response = api_request(
                endpoint,
                params={"page": page, "page_size": page_size}
            )

        except Exception as e:
            print(f"Error on {endpoint} page {page}: {e}")
            print("Retrying page...")
            time.sleep(5)
            continue

        records = response["data"]

        if not records:
            break

        all_records.extend(records)

        print(f"{endpoint} page {page} -> {len(records)} records")

        page += 1

    df = pd.DataFrame(all_records)

    df["_ingested_at"] = datetime.utcnow()
    df["_source_endpoint"] = endpoint

    output_file = BRONZE_PATH / f"{endpoint}.csv"

    df.to_csv(output_file, index=False)

    print(f"{endpoint} saved to {output_file} ({len(df)} rows)")

In [22]:
ingest_endpoint("orders")

orders page 1 -> 1000 records
orders page 2 -> 1000 records
orders page 3 -> 1000 records
orders page 4 -> 1000 records
Retrying in 1s (status 500)
Retrying in 2s (status 429)
orders page 5 -> 1000 records
orders page 6 -> 1000 records
orders page 7 -> 1000 records
orders page 8 -> 1000 records
orders page 9 -> 1000 records
orders page 10 -> 1000 records
Retrying in 1s (status 500)
orders page 11 -> 1000 records
orders page 12 -> 1000 records
Retrying in 1s (status 429)
Retrying in 2s (status 429)
orders page 13 -> 1000 records
orders page 14 -> 1000 records
Retrying in 1s (status 500)
Retrying in 2s (status 500)
orders page 15 -> 1000 records
orders page 16 -> 1000 records
orders page 17 -> 1000 records
orders page 18 -> 1000 records
orders page 19 -> 1000 records
Retrying in 1s (status 429)
Retrying in 2s (status 429)
orders page 20 -> 1000 records
orders page 21 -> 1000 records
orders page 22 -> 1000 records
orders page 23 -> 1000 records
orders page 24 -> 1000 records
orders page 2

In [25]:
ENDPOINTS = [
    "orders",
    "order_items",
    "customers",
    "products",
    "sellers",
    "payments"
]

for endpoint in ENDPOINTS:
    ingest_endpoint(endpoint)

orders page 1 -> 1000 records
orders page 2 -> 1000 records
orders page 3 -> 1000 records
Retrying in 1s (status 500)
orders page 4 -> 1000 records
orders page 5 -> 1000 records
orders page 6 -> 1000 records
Retrying in 1s (status 429)
Retrying in 2s (status 429)
orders page 7 -> 1000 records
orders page 8 -> 1000 records
orders page 9 -> 1000 records
orders page 10 -> 1000 records
orders page 11 -> 1000 records
orders page 12 -> 1000 records
orders page 13 -> 1000 records
orders page 14 -> 1000 records
orders page 15 -> 1000 records
Retrying in 1s (status 429)
Retrying in 2s (status 429)
orders page 16 -> 1000 records
orders page 17 -> 1000 records
orders page 18 -> 1000 records
orders page 19 -> 1000 records
orders page 20 -> 1000 records
orders page 21 -> 1000 records
orders page 22 -> 1000 records
orders page 23 -> 1000 records
orders page 24 -> 1000 records
Retrying in 1s (status 429)
Retrying in 2s (status 429)
orders page 25 -> 1000 records
orders page 26 -> 1000 records
orders 